# PigeonGraph: Universal Repository Analyzer & Interactive Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hoppy-Beast/pigeongraph/blob/main/assets/pigeongraph_demo.ipynb)

Welcome to the interactive demonstration for **PigeonGraph**, a code knowledge graph and query engine.

This notebook lets you test PigeonGraph against **any public GitHub repository** (such as FastAPI, Django, Gin, Express, Zustand, or your own custom repository) directly in Google Colab without installing local editor extensions.

### What this notebook demonstrates
1. Environment Setup (Node.js 24 LTS via NodeSource)
2. Global PigeonGraph Installation from GitHub
3. Interactive Repository Configuration (FastAPI, Django, Gin, Express, Zustand, or any custom Git URL)
4. Automated Project Indexing (`pigeongraph init`)
5. Single-Turn Architecture Exploration (`pigeongraph explore`)
6. Interface Invariant Blast Radius Assessment
7. Clear Visual Architecture Dashboard displaying exact symbols, line ranges, and source code spans

## 1. Environment setup

PigeonGraph requires **Node.js >= 22.5.0** (Node 24 LTS recommended for built-in `node:sqlite`).
Colab runtimes default to older Node versions, so we install Node 24 LTS via NodeSource.

In [ ]:
# Install Node.js 24 LTS via NodeSource
!curl -fsSL https://deb.nodesource.com/setup_24.x | sudo -E bash - > /dev/null 2>&1
!sudo apt-get install -y nodejs > /dev/null 2>&1

# Confirm version
!node -v && npm -v


## 2. Install PigeonGraph from GitHub

Clone the repository and run `npm run setup`. This installs dependencies, compiles all packages with `tsc -b`, and links the global `pigeongraph` CLI command.

In [ ]:
%%bash
# Clone PigeonGraph from GitHub and build globally
rm -rf /content/pigeongraph
git clone https://github.com/Hoppy-Beast/pigeongraph.git /content/pigeongraph
cd /content/pigeongraph
npm run setup

# Verify CLI
pigeongraph --help


## 3. Configure target repository

Select a preset open-source repository or enter any custom GitHub repository URL and symbol name to test.

In [ ]:
# @title 🎯 Repository & Query Configuration
# @markdown Choose a preset or set to **Custom** to test any repository:

PRESET = "FastAPI (Python)"  # @param ["FastAPI (Python)", "Django (Python)", "Gin (Go)", "Express (Node.js)", "Zustand (TypeScript)", "Custom"]
CUSTOM_REPO_URL = "https://github.com/fastapi/fastapi.git"  # @param {type:"string"}
CUSTOM_QUERY_SYMBOL = "solve_dependencies"  # @param {type:"string"}

PRESET_CONFIGS = {
    "FastAPI (Python)": {
        "url": "https://github.com/fastapi/fastapi.git",
        "symbol": "solve_dependencies",
        "description": "Dependency injection resolution and lifecycle engine"
    },
    "Django (Python)": {
        "url": "https://github.com/django/django.git",
        "symbol": "get_response",
        "description": "WSGI/ASGI request handler and middleware dispatch"
    },
    "Gin (Go)": {
        "url": "https://github.com/gin-gonic/gin.git",
        "symbol": "handleHTTPRequest",
        "description": "Core HTTP router engine and middleware pipeline"
    },
    "Express (Node.js)": {
        "url": "https://github.com/expressjs/express.git",
        "symbol": "handle",
        "description": "HTTP request router and middleware execution"
    },
    "Zustand (TypeScript)": {
        "url": "https://github.com/pmndrs/zustand.git",
        "symbol": "createStore",
        "description": "Reactive state store creation and subscriber manager"
    }
}

if PRESET == "Custom":
    TARGET_URL = CUSTOM_REPO_URL.strip()
    TARGET_SYMBOL = CUSTOM_QUERY_SYMBOL.strip()
    TARGET_DESC = "User-configured custom repository"
else:
    cfg = PRESET_CONFIGS[PRESET]
    TARGET_URL = cfg["url"]
    TARGET_SYMBOL = cfg["symbol"]
    TARGET_DESC = cfg["description"]

print(f"Target Repository : {TARGET_URL}")
print(f"Query Symbol      : {TARGET_SYMBOL}")
print(f"Description       : {TARGET_DESC}")


## 4. Automated clone, initialization, and exploration

This cell executes the automated testing pipeline:
1. Clones the target repository (shallow clone)
2. Runs `pigeongraph init` to generate configuration and agent MCP descriptors
3. Executes `pigeongraph explore "<symbol>"` and measures execution latency
4. Runs PR blast radius audit

In [ ]:
import os
import subprocess
import json
import time

TARGET_DIR = "/content/target-repo"

# 1. Clone target repository
print(f"📦 Cloning {TARGET_URL} (shallow clone)...")
subprocess.run(["rm", "-rf", TARGET_DIR], check=True)
subprocess.run(["git", "clone", "--depth", "1", TARGET_URL, TARGET_DIR], check=True)

# 2. Initialize PigeonGraph in the target project
print("\n⚙️ Running 'pigeongraph init'...")
init_res = subprocess.run(["pigeongraph", "init"], cwd=TARGET_DIR, capture_output=True, text=True)
print(init_res.stdout.strip())

# 3. Execute 1-shot exploration
print(f"\n⚡ Querying knowledge graph for '{TARGET_SYMBOL}'...")
start_time = time.time()
explore_res = subprocess.run(
    ["pigeongraph", "explore", TARGET_SYMBOL],
    cwd=TARGET_DIR,
    capture_output=True,
    text=True
)
total_wall_ms = round((time.time() - start_time) * 1000, 2)

if explore_res.returncode != 0:
    print(f"⚠️ Exploration stderr: {explore_res.stderr}")
    explore_data = {}
else:
    try:
        explore_data = json.loads(explore_res.stdout)
    except Exception as e:
        print(f"⚠️ JSON parse error: {e}\nRaw output:\n{explore_res.stdout}")
        explore_data = {}

print(f"\n✅ Exploration completed in {total_wall_ms}ms!")


## 5. Main outputs and architecture dashboard

The dashboard below presents the primary outputs in a clean visual layout: latency, discovered symbols, line coordinates, blast-radius risk level, and extracted source code spans.

In [ ]:
from IPython.display import display, HTML

summary = explore_data.get("query_summary", {})
symbols = explore_data.get("symbols", [])
flows = explore_data.get("execution_flows", {})
dispatches = explore_data.get("dynamic_dispatches", [])
blast = explore_data.get("blast_radius", {})
spans = explore_data.get("served_spans", [])

risk_level = blast.get("risk_level", "LOW")
risk_color = "#1a7f37" if risk_level == "LOW" else "#b08800" if risk_level in ("MEDIUM", "MODERATE") else "#cf222e"
risk_bg = "#dafbe1" if risk_level == "LOW" else "#fff8c5" if risk_level in ("MEDIUM", "MODERATE") else "#ffebe9"

# Format symbol rows
symbols_rows = ""
for s in symbols:
    lines = s.get("lineRange", [0, 0])
    symbols_rows += f"""
    <tr style="border-bottom: 1px solid #e1e4e8;">
      <td style="padding: 8px 12px; font-weight: 600;"><code>{s.get('name')}</code></td>
      <td style="padding: 8px 12px;"><span style="background: #eef1f4; padding: 2px 6px; border-radius: 4px; font-size: 12px;">{s.get('kind')}</span></td>
      <td style="padding: 8px 12px;"><code>{s.get('filePath')}</code></td>
      <td style="padding: 8px 12px; color: #57606a;">L{lines[0]}–L{lines[1]}</td>
    </tr>
    """

if not symbols_rows:
    symbols_rows = "<tr><td colspan='4' style='padding: 12px; text-align: center; color: #586069;'>No exact symbol match discovered for this query.</td></tr>"

# Format dynamic dispatches
dispatch_items = ""
for d in dispatches:
    dispatch_items += f"<li><b>{d.get('pattern', 'DISPATCH')}:</b> <code>{d.get('emitter', '')}</code> &rarr; <code>{d.get('listener', '')}</code></li>"
if not dispatch_items:
    dispatch_items = "<li>No dynamic dispatch channels registered for this symbol.</li>"

# Format extracted code span
code_content = spans[0].get("content", "No source span available") if spans else "No source span available"
repo_shortname = TARGET_URL.split('/')[-1].replace('.git', '')

dashboard_html = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; max-width: 850px; margin: 10px 0; color: #24292e;">
  <!-- Header Card -->
  <div style="padding: 20px; border: 1px solid #d0d7de; border-radius: 8px 8px 0 0; background: linear-gradient(180deg, #f6f8fa 0%, #ffffff 100%);">
    <div style="display: flex; justify-content: space-between; align-items: center;">
      <div>
        <h2 style="margin: 0 0 6px 0; color: #0969da; display: flex; align-items: center; gap: 8px;">
          🐦 PigeonGraph Architecture Report
        </h2>
        <div style="color: #57606a; font-size: 14px;">
          Repository: <b>{repo_shortname}</b> &nbsp;|&nbsp; Query: <code>{TARGET_SYMBOL}</code>
        </div>
      </div>
      <div style="text-align: right;">
        <span style="display: inline-block; padding: 4px 12px; border-radius: 12px; font-weight: 600; font-size: 13px; background: {risk_bg}; color: {risk_color};">
          Blast Radius: {risk_level}
        </span>
        <div style="font-size: 12px; color: #57606a; margin-top: 4px;">Risk Score: {blast.get('risk_score', 0.0)}</div>
      </div>
    </div>
  </div>

  <!-- Key Metrics Row -->
  <div style="display: grid; grid-template-columns: repeat(4, 1fr); border-left: 1px solid #d0d7de; border-right: 1px solid #d0d7de; border-bottom: 1px solid #d0d7de; background: #ffffff;">
    <div style="padding: 12px 16px; border-right: 1px solid #eaecef; text-align: center;">
      <div style="font-size: 11px; color: #57606a; text-transform: uppercase; font-weight: 600;">Latency</div>
      <div style="font-size: 20px; font-weight: bold; color: #0969da; margin-top: 2px;">{summary.get('duration_ms', 0)} ms</div>
    </div>
    <div style="padding: 12px 16px; border-right: 1px solid #eaecef; text-align: center;">
      <div style="font-size: 11px; color: #57606a; text-transform: uppercase; font-weight: 600;">Nodes Searched</div>
      <div style="font-size: 20px; font-weight: bold; color: #24292e; margin-top: 2px;">{summary.get('total_graph_nodes_searched', 0)}</div>
    </div>
    <div style="padding: 12px 16px; border-right: 1px solid #eaecef; text-align: center;">
      <div style="font-size: 11px; color: #57606a; text-transform: uppercase; font-weight: 600;">Affected Files</div>
      <div style="font-size: 20px; font-weight: bold; color: #24292e; margin-top: 2px;">{blast.get('affected_files_count', 0)}</div>
    </div>
    <div style="padding: 12px 16px; text-align: center;">
      <div style="font-size: 11px; color: #57606a; text-transform: uppercase; font-weight: 600;">Epistemic Status</div>
      <div style="font-size: 15px; font-weight: bold; color: #1a7f37; margin-top: 5px;">{summary.get('epistemic_status', 'N/A')}</div>
    </div>
  </div>

  <!-- Discovered Symbols Section -->
  <div style="padding: 16px 20px; border-left: 1px solid #d0d7de; border-right: 1px solid #d0d7de; border-bottom: 1px solid #d0d7de; background: #ffffff;">
    <h4 style="margin: 0 0 10px 0;">Discovered Symbols ({len(symbols)})</h4>
    <table style="width: 100%; border-collapse: collapse; font-size: 13px;">
      <thead>
        <tr style="background: #f6f8fa; border-bottom: 2px solid #d0d7de; text-align: left;">
          <th style="padding: 8px 12px;">Symbol Name</th>
          <th style="padding: 8px 12px;">Kind</th>
          <th style="padding: 8px 12px;">File Path</th>
          <th style="padding: 8px 12px;">Coordinates</th>
        </tr>
      </thead>
      <tbody>
        {symbols_rows}
      </tbody>
    </table>
  </div>

  <!-- Dynamic Dispatch Section -->
  <div style="padding: 16px 20px; border-left: 1px solid #d0d7de; border-right: 1px solid #d0d7de; border-bottom: 1px solid #d0d7de; background: #ffffff;">
    <h4 style="margin: 0 0 8px 0;">Dynamic Dispatch Channels</h4>
    <ul style="margin: 0; padding-left: 20px; font-size: 13px; color: #24292e;">
      {dispatch_items}
    </ul>
  </div>

  <!-- Extracted Source Code Span -->
  <div style="padding: 16px 20px; border: 1px solid #d0d7de; border-top: none; border-radius: 0 0 8px 8px; background: #ffffff;">
    <h4 style="margin: 0 0 8px 0;">Extracted Source Code Span</h4>
    <pre style="margin: 0; padding: 12px; background: #f6f8fa; border: 1px solid #e1e4e8; border-radius: 6px; font-size: 12px; overflow-x: auto; font-family: monospace; white-space: pre-wrap;">{code_content}</pre>
  </div>
</div>
"""

display(HTML(dashboard_html))
